# MultiVI FAP regulatory analyses

This notebook collects the code used for the MultiVI FAP analyses in `mutivi_results`.
It is organized as reusable cells so individual figures can be regenerated without keeping separate scripts.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import re
import json
import math
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr, rankdata
from scipy.stats import t as tdist
from statsmodels.nonparametric.smoothers_lowess import lowess

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import muon as mu
import scanpy as sc
import snapatac2 as snap

warnings.filterwarnings("ignore")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

BASE_DIR = Path("/Users/mingkewu/Documents/skeletal/mutivi/cornell+bch")
H5MU = BASE_DIR / "fap_multivi_orig_ident_complete.h5mu"
OUT_DIR = BASE_DIR / "mutivi_results"
META_XLSX = Path("/Users/mingkewu/Documents/skeletal/Ped_SkM_Sample_Meta.xlsx")
GFF = Path("/Users/mingkewu/Library/Caches/snapatac2/gencode_v41_GRCh38.gff3.gz")
GENOME_FASTA = OUT_DIR / "GRCh38.primary_assembly.genome.fa"

OUT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTERS = ["MME+", "LUM+", "CD55+", "GPC3+", "COL11A1+", "ACTA1+"]
CLUSTER_COLORS = {
    "MME+": "#4C78A8",
    "LUM+": "#F58518",
    "CD55+": "#54A24B",
    "GPC3+": "#B279A2",
    "COL11A1+": "#E45756",
    "ACTA1+": "#72B7B2",
}
LEIDEN_TO_SUBTYPE = {
    "0": "CD55+",
    "2": "MME+", "3": "MME+", "4": "MME+", "5": "MME+",
    "7": "LUM+",
    "1": "GPC3+", "8": "GPC3+", "10": "GPC3+", "11": "GPC3+",
    "9": "COL11A1+",
    "6": "ACTA1+",
}


In [ ]:
NS = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


def xlsx_col_to_idx(ref):
    letters = "".join(ch for ch in ref if ch.isalpha())
    idx = 0
    for ch in letters:
        idx = idx * 26 + ord(ch.upper()) - 64
    return idx - 1


def read_xlsx_first_sheet(path):
    with ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", NS):
                shared.append("".join(t.text or "" for t in si.findall(".//a:t", NS)))
        sheet = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in sheet.findall(".//a:sheetData/a:row", NS):
            values = {}
            for cell in row.findall("a:c", NS):
                idx = xlsx_col_to_idx(cell.attrib.get("r", "A"))
                typ = cell.attrib.get("t")
                node = cell.find("a:v", NS)
                val = "" if node is None else (node.text or "")
                if typ == "s" and val:
                    val = shared[int(val)]
                elif typ == "inlineStr":
                    val = "".join(t.text or "" for t in cell.findall(".//a:t", NS))
                values[idx] = val
            if values:
                rows.append([values.get(i, "") for i in range(max(values) + 1)])
    width = max(len(row) for row in rows)
    rows = [row + [""] * (width - len(row)) for row in rows]
    return pd.DataFrame(rows[1:], columns=rows[0])


def parse_age_years(value):
    text = str(value).strip().lower()
    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", text)
    if not match:
        return np.nan
    age = float(match.group(1))
    return age / 12.0 if "mo" in text or "month" in text else age


def donor_candidates(orig_ident):
    base = str(orig_ident).split("_")[0]
    candidates = [base]
    if base.startswith("CTRL"):
        stripped = re.sub(r"([a-z]+)$", "", base)
        if stripped != base:
            candidates.append(stripped)
    if base.isdigit() and len(base) > 6:
        candidates.extend([base[:-1], base[:6]])
    return list(dict.fromkeys(candidates))


def bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full(pvalues.shape, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    vals = pvalues[valid]
    if vals.size == 0:
        return out
    order = np.argsort(vals)
    ranks = np.arange(1, vals.size + 1)
    q = vals[order] * vals.size / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    tmp = np.empty_like(vals)
    tmp[order] = np.clip(q, 0, 1)
    out[valid] = tmp
    return out


def pseudobulk_mean(X, groups):
    groups = np.asarray(groups)
    cats = pd.Categorical(groups)
    codes = cats.codes
    matrix = sp.csr_matrix(
        (np.ones(len(groups)), (codes, np.arange(len(groups)))),
        shape=(len(cats.categories), len(groups)),
    )
    sums = matrix @ X
    counts = np.asarray(matrix.sum(axis=1)).ravel()
    if sp.issparse(sums):
        sums = sums.toarray()
    return sums / counts[:, None], np.asarray(cats.categories), counts


def spearman_vec(age, mat):
    rhos, pvals = [], []
    for j in range(mat.shape[1]):
        values = mat[:, j]
        if np.nanstd(values) == 0 or len(np.unique(age)) < 3:
            rhos.append(np.nan)
            pvals.append(np.nan)
        else:
            rho, pval = spearmanr(age, values, nan_policy="omit")
            rhos.append(rho)
            pvals.append(pval)
    return np.asarray(rhos), np.asarray(pvals)


def spearman_chunked(X, age, chunk=400):
    n = X.shape[0]
    ranked_age = rankdata(age, method="average").astype(np.float64)
    ranked_age -= ranked_age.mean()
    age_norm = np.sqrt(np.sum(ranked_age * ranked_age))
    rhos = np.full(X.shape[1], np.nan)
    pvals = np.full(X.shape[1], np.nan)
    for start in range(0, X.shape[1], chunk):
        end = min(start + chunk, X.shape[1])
        sub = X[:, start:end]
        arr = sub.toarray() if sp.issparse(sub) else np.asarray(sub)
        arr = arr.astype(np.float64, copy=False)
        ranks = np.apply_along_axis(rankdata, 0, arr, method="average")
        ranks -= ranks.mean(axis=0)
        denom = age_norm * np.sqrt(np.sum(ranks * ranks, axis=0))
        rho = np.divide(ranked_age @ ranks, denom, out=np.full(end - start, np.nan), where=denom > 0)
        rho = np.clip(rho, -0.999999, 0.999999)
        t_stat = rho * np.sqrt((n - 2) / np.maximum(1e-12, 1 - rho * rho))
        pval = 2 * tdist.sf(np.abs(t_stat), df=n - 2)
        rhos[start:end] = rho
        pvals[start:end] = pval
    return rhos, pvals


def timing_t50(age, values, direction):
    age = np.asarray(age, dtype=float)
    values = np.asarray(values, dtype=float)
    ok = np.isfinite(age) & np.isfinite(values)
    age = age[ok]
    values = values[ok]
    if len(np.unique(age)) < 5 or len(values) < 6:
        return np.nan, np.nan, None
    order = np.argsort(age)
    age = age[order]
    values = values[order] * direction
    try:
        smooth = lowess(values, age, frac=0.65, it=1, return_sorted=False)
    except Exception:
        smooth = values
    dynamic_range = float(np.nanpercentile(smooth, 90) - np.nanpercentile(smooth, 10))
    if not np.isfinite(dynamic_range) or dynamic_range <= 0:
        return np.nan, dynamic_range, smooth
    target = np.nanmin(smooth) + 0.5 * (np.nanmax(smooth) - np.nanmin(smooth))
    hits = np.where(smooth >= target)[0]
    if hits.size == 0:
        return np.nan, dynamic_range, smooth
    return float(age[hits[0]]), dynamic_range, smooth


def zscore(values):
    values = np.asarray(values, dtype=float)
    sd = np.nanstd(values)
    return (values - np.nanmean(values)) / (sd if sd > 0 else 1)


In [ ]:
def load_multivi_fap_data(min_cells_per_donor_cluster=20):
    meta = read_xlsx_first_sheet(META_XLSX)
    meta["donor_id"] = meta["Donor ID"].astype(str)
    meta["age_num"] = meta["Age"].map(parse_age_years)
    age_lookup = meta.set_index("donor_id")["age_num"].to_dict()

    mdata = mu.read_h5mu(H5MU)
    rna = mdata.mod["rna"].copy()
    atac = mdata.mod["atac"].copy()

    rna.obsm["X_multivi"] = np.asarray(mdata.obsm["X_multivi"])
    sc.pp.neighbors(rna, n_neighbors=15, use_rep="X_multivi", metric="euclidean", random_state=0)
    sc.tl.leiden(
        rna,
        resolution=0.7,
        flavor="igraph",
        n_iterations=2,
        directed=False,
        random_state=0,
        key_added="leiden_fap_multivi_res0.7",
    )
    subtype = rna.obs["leiden_fap_multivi_res0.7"].astype(str).map(LEIDEN_TO_SUBTYPE)

    donors, ages = [], []
    for sample in rna.obs["orig.ident"].astype(str):
        donor = next((cand for cand in donor_candidates(sample) if cand in age_lookup), None)
        donors.append(donor)
        ages.append(age_lookup.get(donor, np.nan))
    donors = np.asarray(donors, dtype=object)
    ages = np.asarray(ages, dtype=float)

    mask = (
        rna.obs["has_atac"].astype(str).isin(["True", "true", "1"]).values
        & rna.obs["has_rna"].astype(str).isin(["True", "true", "1"]).values
        & subtype.notna().values
        & np.isfinite(ages)
        & pd.notna(donors)
    )
    rna = rna[mask].copy()
    atac = atac[mask].copy()
    subtype = subtype[mask].astype(str).values
    donors = donors[mask]
    ages = ages[mask]

    rna.obs["fap_subtype"] = pd.Categorical(subtype, categories=CLUSTERS)
    rna.obs["donor_id_for_age"] = donors
    rna.obs["age_num"] = ages
    atac.obs["fap_subtype"] = rna.obs["fap_subtype"].copy()
    atac.obs["donor_id_for_age"] = donors
    atac.obs["age_num"] = ages

    print("Cells used:", rna.n_obs)
    print(pd.Series(subtype).value_counts().reindex(CLUSTERS).to_string())
    return rna, atac, subtype, donors, ages, age_lookup


rna, atac, subtype, donors, ages, age_lookup = load_multivi_fap_data()


In [ ]:
def make_gene_activity_matrix(atac):
    tmp = atac.copy()
    tmp.X = tmp.X.astype("uint32") if sp.issparse(tmp.X) else tmp.X.astype("uint32")
    gene_activity = snap.pp.make_gene_matrix(
        tmp,
        GFF,
        use_x=True,
        inplace=False,
        backend=None,
        id_type="gene",
        upstream=2000,
        downstream=0,
        include_gene_body=True,
    )
    return gene_activity


gene_activity = make_gene_activity_matrix(atac)
print(gene_activity.shape)


In [ ]:
def align_rna_and_gene_activity(rna, gene_activity):
    common = sorted(set(rna.var_names.astype(str)).intersection(set(gene_activity.var_names.astype(str))))
    rna_index = pd.Index(rna.var_names.astype(str))
    activity_index = pd.Index(gene_activity.var_names.astype(str))
    rna_idx = [rna_index.get_loc(gene) for gene in common]
    activity_idx = [activity_index.get_loc(gene) for gene in common]
    rna_aligned = rna[:, rna_idx].copy()
    activity_aligned = gene_activity[:, activity_idx].copy()
    rna_aligned.var_names = common
    activity_aligned.var_names = common
    sc.pp.normalize_total(rna_aligned, target_sum=1e4)
    sc.pp.log1p(rna_aligned)
    sc.pp.normalize_total(activity_aligned, target_sum=1e4)
    sc.pp.log1p(activity_aligned)
    return rna_aligned, activity_aligned, common


rna_aligned, activity_aligned, common_genes = align_rna_and_gene_activity(rna, gene_activity)
print(len(common_genes))


In [ ]:
def cell_level_activity_expression_overlap(rna_aligned, activity_aligned, subtype, ages, q_thresholds=(0.05, 0.1, 0.5)):
    rows = []
    examples = {}
    for cluster in CLUSTERS:
        idx = np.where(subtype == cluster)[0]
        age = ages[idx].astype(float)
        expr_rho, expr_p = spearman_chunked(rna_aligned.X[idx, :], age, chunk=350)
        act_rho, act_p = spearman_chunked(activity_aligned.X[idx, :], age, chunk=350)
        expr_q = bh_fdr(expr_p)
        act_q = bh_fdr(act_p)
        same_sign = np.sign(expr_rho) == np.sign(act_rho)
        opposite_sign = np.sign(expr_rho) == -np.sign(act_rho)
        for q_threshold in q_thresholds:
            both = (expr_q < q_threshold) & (act_q < q_threshold) & np.isfinite(expr_rho) & np.isfinite(act_rho)
            same = both & same_sign
            opposite = both & opposite_sign
            rows.append({
                "cluster": cluster,
                "q_threshold": q_threshold,
                "tested_common_genes": len(common_genes),
                "expr_sig": int(np.sum(expr_q < q_threshold)),
                "activity_sig": int(np.sum(act_q < q_threshold)),
                "both_sig": int(np.sum(both)),
                "same_direction": int(np.sum(same)),
                "opposite_direction": int(np.sum(opposite)),
                "same_up": int(np.sum(same & (expr_rho > 0) & (act_rho > 0))),
                "same_down": int(np.sum(same & (expr_rho < 0) & (act_rho < 0))),
            })
        both05 = (expr_q < 0.05) & (act_q < 0.05) & same_sign & np.isfinite(expr_rho) & np.isfinite(act_rho)
        score = (np.abs(expr_rho) + np.abs(act_rho)) / 2
        top_idx = np.where(both05)[0]
        top_idx = top_idx[np.argsort(score[top_idx])[::-1][:12]] if top_idx.size else []
        examples[cluster] = [f"{common_genes[i]}({'+' if expr_rho[i] > 0 else '-'})" for i in top_idx]
    return pd.DataFrame(rows), examples


overlap_table, overlap_examples = cell_level_activity_expression_overlap(rna_aligned, activity_aligned, subtype, ages)
print(overlap_table.to_string(index=False))
print(overlap_examples)


In [ ]:
def donor_level_activity_leads_expression(rna_aligned, activity_aligned, subtype, donors, ages, age_lookup, q_threshold=0.1, min_cells=20):
    records = []
    plot_data = {}
    for cluster in CLUSTERS:
        cell_idx = np.where(subtype == cluster)[0]
        donor_counts = pd.Series(donors[cell_idx]).value_counts()
        keep_donors = donor_counts[donor_counts >= min_cells].index.astype(object).tolist()
        idx = cell_idx[np.isin(donors[cell_idx], keep_donors)]
        if len(keep_donors) < 6:
            continue
        expr_pb, donor_order, _ = pseudobulk_mean(rna_aligned.X[idx, :], donors[idx])
        act_pb, donor_order2, _ = pseudobulk_mean(activity_aligned.X[idx, :], donors[idx])
        assert list(donor_order) == list(donor_order2)
        donor_age = np.array([age_lookup[d] for d in donor_order], dtype=float)
        order = np.argsort(donor_age)
        donor_age = donor_age[order]
        expr_pb = expr_pb[order, :]
        act_pb = act_pb[order, :]
        expr_rho, expr_p = spearman_vec(donor_age, expr_pb)
        act_rho, act_p = spearman_vec(donor_age, act_pb)
        expr_q = bh_fdr(expr_p)
        act_q = bh_fdr(act_p)
        same = np.sign(expr_rho) == np.sign(act_rho)
        candidates = np.where((expr_q < q_threshold) & (act_q < q_threshold) & same & np.isfinite(expr_rho) & np.isfinite(act_rho))[0]
        plot_data[cluster] = {
            "age": donor_age,
            "expr": expr_pb,
            "act": act_pb,
            "expr_rho": expr_rho,
            "act_rho": act_rho,
            "expr_q": expr_q,
            "act_q": act_q,
        }
        for j in candidates:
            direction = 1 if (expr_rho[j] + act_rho[j]) >= 0 else -1
            activity_t50, _, _ = timing_t50(donor_age, act_pb[:, j], direction)
            expr_t50, _, _ = timing_t50(donor_age, expr_pb[:, j], direction)
            if np.isfinite(activity_t50) and np.isfinite(expr_t50) and (expr_t50 - activity_t50) >= 1.0:
                records.append({
                    "cluster": cluster,
                    "gene": common_genes[j],
                    "direction": "up" if direction > 0 else "down",
                    "activity_t50": activity_t50,
                    "rna_t50": expr_t50,
                    "lead_years": expr_t50 - activity_t50,
                    "activity_rho": act_rho[j],
                    "rna_rho": expr_rho[j],
                    "activity_q": act_q[j],
                    "rna_q": expr_q[j],
                    "gene_index": j,
                })
    return pd.DataFrame(records).sort_values(["cluster", "lead_years"], ascending=[True, False]).reset_index(drop=True), plot_data


lead_genes, lead_plot_data = donor_level_activity_leads_expression(
    rna_aligned, activity_aligned, subtype, donors, ages, age_lookup, q_threshold=0.1, min_cells=20
)
print(lead_genes[["cluster", "gene", "direction", "lead_years", "activity_rho", "rna_rho", "activity_q", "rna_q"]].to_string(index=False))


In [ ]:
def plot_activity_leads_expression(lead_genes, plot_data, output_path=OUT_DIR / "snapatac2_activity_leads_rna_age_trajectories.pdf"):
    if lead_genes.empty:
        raise ValueError("No genes passed the activity-leading criteria.")
    with PdfPages(output_path) as pdf:
        fig, ax = plt.subplots(figsize=(10.5, max(4.5, 0.32 * len(lead_genes) + 1.7)))
        y = np.arange(len(lead_genes))
        colors = [CLUSTER_COLORS[c] for c in lead_genes["cluster"]]
        ax.barh(y, lead_genes["lead_years"], color=colors, alpha=0.85, edgecolor="none")
        labels = [f"{c}  {g} ({'up' if d == 'up' else 'down'})" for c, g, d in zip(lead_genes.cluster, lead_genes.gene, lead_genes.direction)]
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=7)
        ax.invert_yaxis()
        ax.axvline(1, color="#333333", lw=0.8, ls="--")
        ax.set_xlabel("Activity lead over RNA expression (years)", fontsize=9)
        ax.set_title("Genes with donor-level gene activity preceding RNA expression by at least 1 year", fontsize=12, fontweight="bold")
        ax.grid(axis="x", color="#dddddd", lw=0.6)
        ax.spines[["top", "right", "left"]].set_visible(False)
        handles = [mpl.patches.Patch(color=CLUSTER_COLORS[c], label=c) for c in CLUSTERS if c in set(lead_genes.cluster)]
        ax.legend(handles=handles, frameon=False, fontsize=7, ncol=3, loc="lower right")
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        for cluster in CLUSTERS:
            sub = lead_genes[lead_genes.cluster == cluster].copy()
            if sub.empty:
                continue
            ncols = 3 if len(sub) >= 3 else len(sub)
            nrows = int(np.ceil(len(sub) / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.1 * nrows), squeeze=False)
            data = plot_data[cluster]
            age = data["age"]
            for ax, (_, row) in zip(axes.ravel(), sub.iterrows()):
                j = int(row.gene_index)
                expr_values = zscore(data["expr"][:, j])
                act_values = zscore(data["act"][:, j])
                order = np.argsort(age)
                xs = age[order]
                ax.scatter(age, act_values, s=18, color="#D62728", alpha=0.55, linewidth=0, label="gene activity")
                ax.scatter(age, expr_values, s=18, color="#1F77B4", alpha=0.55, linewidth=0, label="RNA expression")
                ax.plot(xs, lowess(act_values[order], xs, frac=0.65, it=1, return_sorted=False), color="#D62728", lw=2)
                ax.plot(xs, lowess(expr_values[order], xs, frac=0.65, it=1, return_sorted=False), color="#1F77B4", lw=2)
                ax.axvline(row.activity_t50, color="#D62728", lw=1.1, ls="--", alpha=0.8)
                ax.axvline(row.rna_t50, color="#1F77B4", lw=1.1, ls="--", alpha=0.8)
                ax.set_title(f"{row.gene} ({row.direction}, lead={row.lead_years:.1f}y)", fontsize=9, fontweight="bold")
                ax.text(
                    0.03,
                    0.05,
                    f"activity rho={row.activity_rho:.2f}, q={row.activity_q:.2g}\nRNA rho={row.rna_rho:.2f}, q={row.rna_q:.2g}",
                    transform=ax.transAxes,
                    fontsize=6.8,
                    va="bottom",
                )
                ax.set_xlabel("Age (years)", fontsize=8)
                ax.set_ylabel("donor pseudobulk z-score", fontsize=8)
                ax.tick_params(labelsize=7)
                ax.grid(color="#e0e0e0", lw=0.5)
                ax.spines[["top", "right"]].set_visible(False)
            for ax in axes.ravel()[len(sub):]:
                ax.axis("off")
            axes[0, 0].legend(frameon=False, fontsize=7, loc="upper left")
            fig.suptitle(f"{cluster}: activity precedes RNA expression", fontsize=13, fontweight="bold", y=1.01)
            fig.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
    return output_path


plot_activity_leads_expression(lead_genes, lead_plot_data)


In [ ]:
def peak_to_region(name):
    if ":" in name:
        return name
    chrom, start, end = str(name).split("-")[:3]
    return f"{chrom}:{start}-{end}"


def dense_mean(X, idx):
    sub = X[idx]
    return np.asarray(sub.mean(axis=0)).ravel() if sp.issparse(sub) else np.asarray(sub.mean(axis=0)).ravel()


def cluster_specific_regions(atac, subtype, top_n=750, min_mean=0.01):
    X = atac.X.tocsr() if sp.issparse(atac.X) else np.asarray(atac.X)
    peak_names = np.asarray(atac.var_names)
    region_sets = {}
    for cluster in CLUSTERS:
        idx = np.where(subtype == cluster)[0]
        target_mean = dense_mean(X, idx)
        other_means = [dense_mean(X, np.where(subtype == other)[0]) for other in CLUSTERS if other != cluster]
        other_max = np.vstack(other_means).max(axis=0)
        specificity = target_mean - other_max
        valid = (target_mean > min_mean) & (specificity > 0)
        ranked = np.where(valid)[0][np.argsort(specificity[valid])[::-1][:top_n]]
        region_sets[cluster] = [peak_to_region(x) for x in peak_names[ranked]]
    return region_sets


def run_motif_enrichment_by_cluster(atac, subtype, output_path=OUT_DIR / "snapatac2_tf_by_cluster.pdf"):
    regions = cluster_specific_regions(atac, subtype, top_n=750, min_mean=0.01)
    motifs = snap.datasets.cis_bp(unique=True)
    enrichment = snap.tl.motif_enrichment(motifs, regions, genome_fasta=GENOME_FASTA, method="hypergeometric")
    rows = []
    for cluster, table in enrichment.items():
        df = table.to_pandas() if hasattr(table, "to_pandas") else pd.DataFrame(table)
        df["cluster"] = cluster
        rows.append(df)
    motif_table = pd.concat(rows, ignore_index=True)
    score_col = next((c for c in ["minus_log10_p_value", "-log10(p-value)", "score", "log2(fold change)"] if c in motif_table.columns), None)
    name_col = next((c for c in ["name", "motif", "id"] if c in motif_table.columns), None)
    if score_col is None or name_col is None:
        return motif_table
    top = motif_table.sort_values(score_col, ascending=False).groupby("cluster", observed=True).head(5)
    genes = list(dict.fromkeys(top[name_col].astype(str)))
    mat = top.pivot_table(index="cluster", columns=name_col, values=score_col, aggfunc="max").reindex(CLUSTERS).reindex(columns=genes).fillna(0)
    fig, ax = plt.subplots(figsize=(max(7, 0.32 * len(genes)), 3.4))
    im = ax.imshow(mat.values, cmap="Reds", aspect="auto")
    ax.set_xticks(np.arange(len(genes)))
    ax.set_xticklabels(genes, rotation=90, fontsize=6)
    ax.set_yticks(np.arange(len(CLUSTERS)))
    ax.set_yticklabels(CLUSTERS, fontsize=8)
    ax.set_title("Cluster-specific motif enrichment", fontsize=11, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label=score_col)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)
    return motif_table


# motif_table = run_motif_enrichment_by_cluster(atac, subtype)


In [ ]:
MOTIF_BLOCKS = {
    "MME+": ["CTCF", "ZFP64", "EGR1", "ZNF846", "ZNF460", "ZNF311", "ZFP57"],
    "LUM+": ["KLF12", "SP2", "ZNF441", "KLF1", "KLF7", "ZNF777", "NRF1"],
    "CD55+": ["JUN", "JUND", "ATF2", "CREB5", "FOSB", "ATF3", "JDP2"],
    "GPC3+": ["KLF4", "SP1", "WT1", "KLF15", "KLF14", "KLF5", "SP3"],
    "COL11A1+": ["NFATC2", "NFATC4", "TEAD1", "TEAD4", "MYF5"],
    "ACTA1+": ["ZNF502", "ZNF214", "NANOG", "POU2F1", "ZNF582", "ZNF260", "DUX4"],
}

IMPORTANT_GENES = {
    "MME+": ["SMOC2", "ADAM19", "SLCO2B1", "COLEC12", "COL15A1", "COL6A1", "PTGDS", "FADS2", "ITIH5"],
    "LUM+": ["CXCL14", "CRLF1", "GPX3", "HSPG2", "ISLR", "HMCN2", "PTGIS", "SERPINF1", "ADAMTSL2"],
    "CD55+": ["CD55", "CREB5", "TRIO", "MFAP5", "SPSB1", "SOCS3", "HEG1", "KLF4", "TNS1"],
    "GPC3+": ["GPC3", "CNTN4", "FBLN1", "NRP1", "PTCH2", "CDON", "NAV1", "DHRS3", "PTCH1"],
    "COL11A1+": ["COL11A2", "THBS4", "PLEKHG5", "MYLK", "COL1A1", "KCNMA1", "COL6A1", "MEGF6", "SEMA3B"],
    "ACTA1+": ["ACTA1", "TTN", "RCSD1", "RYR1", "ZEB2", "ASB15", "COL4A4", "COL4A3", "S100A11"],
}


def regulatory_network_tf_gene_pairs(rna_aligned, atac, subtype, top_regions=750, cor_threshold=0.03):
    motif_names = sorted(set(sum(MOTIF_BLOCKS.values(), [])))
    motif_set = {x.upper() for x in motif_names}
    name_to_motif = {motif.name.upper(): motif for motif in snap.datasets.cis_bp(unique=True)}
    selected_motifs = [name_to_motif[name.upper()] for name in motif_names if name.upper() in name_to_motif]
    region_sets = cluster_specific_regions(atac, subtype, top_n=top_regions, min_mean=0.01)
    records = []
    for cluster in CLUSTERS:
        print(cluster)
        regions = region_sets[cluster]
        region_to_peak = {peak_to_region(name): name for name in atac.var_names}
        peak_names = [region_to_peak[r] for r in regions if r in region_to_peak]
        peak_sub = atac[:, peak_names].copy()
        peak_sub.var_names = [peak_to_region(x) for x in peak_names]
        network = snap.tl.init_network_from_annotation(
            list(peak_sub.var_names),
            GFF,
            upstream=250000,
            downstream=250000,
            id_type="gene_name",
            coding_gene_only=True,
        )
        snap.tl.add_cor_scores(network, gene_mat=rna_aligned, peak_mat=peak_sub, overwrite=True)
        snap.tl.add_tf_binding(network, motifs=selected_motifs, genome_fasta=GENOME_FASTA, pvalue=1e-4)
        region_gene = []
        tf_region = []
        for edge_index in network.edge_indices():
            source, target = network.get_edge_endpoints_by_index(edge_index)
            source_node = network[source]
            target_node = network[target]
            edge_data = network.get_edge_data_by_index(edge_index)
            cor = getattr(edge_data, "cor_score", None)
            cor = 0 if cor is None else float(cor)
            if source_node.type == "region" and target_node.type == "gene" and cor > cor_threshold:
                region_gene.append((source_node.id, target_node.id, cor))
            elif source_node.type == "gene" and target_node.type == "region" and cor > cor_threshold:
                region_gene.append((target_node.id, source_node.id, cor))
            if source_node.type == "motif" and target_node.type == "region" and source_node.id.upper() in motif_set:
                tf_region.append((source_node.id.upper(), target_node.id))
            elif target_node.type == "motif" and source_node.type == "region" and target_node.id.upper() in motif_set:
                tf_region.append((target_node.id.upper(), source_node.id))
        rg = pd.DataFrame(region_gene, columns=["region", "gene", "cor"])
        tr = pd.DataFrame(tf_region, columns=["tf", "region"])
        if rg.empty or tr.empty:
            continue
        merged = tr.merge(rg, on="region", how="inner")
        if merged.empty:
            continue
        agg = merged.groupby(["tf", "gene"]).agg(
            n_peak_links=("region", "nunique"),
            sum_pos_cor=("cor", "sum"),
        ).reset_index()
        agg["support"] = agg["sum_pos_cor"] * np.log1p(agg["n_peak_links"])
        agg["cluster"] = cluster
        records.append(agg)
    return pd.concat(records, ignore_index=True)


# tf_gene_pairs = regulatory_network_tf_gene_pairs(rna_aligned, atac, subtype)


In [ ]:
def plot_tf_gene_pair_heatmap(tf_gene_pairs, output_path=OUT_DIR / "snapatac2_regulatory_network_cluster_tf_gene_pair_heatmap.pdf"):
    rows = [(cluster, tf) for cluster in CLUSTERS for tf in MOTIF_BLOCKS[cluster]]
    cols = []
    for cluster in CLUSTERS:
        sub = tf_gene_pairs[(tf_gene_pairs.cluster == cluster) & (tf_gene_pairs.tf.isin(MOTIF_BLOCKS[cluster]))]
        score = sub.groupby("gene")["support"].sum().sort_values(ascending=False)
        chosen = []
        for gene in IMPORTANT_GENES[cluster]:
            if gene in score.index and gene not in chosen:
                chosen.append(gene)
        for gene in score.index:
            if gene not in chosen:
                chosen.append(gene)
            if len(chosen) >= 90:
                break
        cols.extend([(cluster, gene) for gene in chosen])
    mat = np.zeros((len(rows), len(cols)))
    row_index = {row: i for i, row in enumerate(rows)}
    col_index = {col: j for j, col in enumerate(cols)}
    for _, row in tf_gene_pairs.iterrows():
        rr = (row.cluster, row.tf)
        cc = (row.cluster, row.gene)
        if rr in row_index and cc in col_index:
            mat[row_index[rr], col_index[cc]] = max(mat[row_index[rr], col_index[cc]], row.support)
    cap = np.nanpercentile(mat[mat > 0], 99) if np.any(mat > 0) else 1
    plot = np.clip(mat / cap, 0, 1)

    fig, ax = plt.subplots(figsize=(max(24, 7 + len(cols) * 0.018), max(12, 5 + len(rows) * 0.055)))
    im = ax.imshow(plot, aspect="auto", cmap="Reds", interpolation="nearest", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks(np.arange(len(rows)))
    ax.set_yticklabels([tf for _, tf in rows], fontsize=4.2)
    ax.tick_params(axis="y", length=0, pad=1.5)
    row_start = 0
    for cluster in CLUSTERS:
        n = len(MOTIF_BLOCKS[cluster])
        ax.axhline(row_start - 0.5, color="#555555", lw=0.55)
        ax.text(-0.047 * len(cols), row_start + n / 2 - 0.5, cluster, ha="right", va="center", fontsize=8, fontweight="bold", clip_on=False)
        row_start += n
    ax.axhline(row_start - 0.5, color="#555555", lw=0.55)
    col_start = 0
    for cluster in CLUSTERS:
        n = sum(1 for col in cols if col[0] == cluster)
        ax.axvline(col_start - 0.5, color="#555555", lw=0.6)
        ax.text(col_start + n / 2 - 0.5, -3.2, cluster, ha="center", va="bottom", fontsize=8, fontweight="bold", clip_on=False)
        col_start += n
    ax.axvline(col_start - 0.5, color="#555555", lw=0.6)
    ax.set_title("SnapATAC2 regulatory network: cluster-specific TF-gene pair support", fontsize=11, fontweight="bold", pad=42)
    fig.colorbar(im, ax=ax, fraction=0.014, pad=0.012, label="TF-gene pair support\n(scaled within plot)")
    fig.savefig(output_path, dpi=450, bbox_inches="tight")
    plt.close(fig)
    return output_path


# plot_tf_gene_pair_heatmap(tf_gene_pairs)
